# Week 1 · Agentic AI와 LangGraph의 첫걸음

**경영정보학과 APPS 학회 · 파이썬 기초 학습자용**

이번 주에는 모델 API를 호출하지 않습니다. 같은 입력에는 같은 결과가 나오는 **결정론적 대학 학생지원 문의 workflow**를 직접 만들며 LangGraph의 뼈대를 익힙니다.

> 이번 주의 성공 기준은 코드를 외우는 것이 아닙니다. 각 단계에서 **누가 다음 행동을 결정하는지**, 그리고 **Node 실행 전후에 State의 어떤 값이 달라지는지**를 자기 말로 설명하는 것입니다.

노트북을 위에서부터 `Shift + Enter`로 실행하세요. 코드 셀의 출력은 환경과 무관하게 재현할 수 있으며 OpenAI 키나 검색 서비스 키가 필요하지 않습니다.

## 오늘의 학습 목표

학습을 마치면 다음을 할 수 있어야 합니다.

1. LLM, workflow, agent의 차이와 agentic AI 시스템의 구조를 설명한다.
2. 작은 `if`문으로 충분한 문제와 그래프로 표현하면 좋은 문제를 구분한다.
3. `State`, `Node`, `Edge`, `START`, `END`를 학생지원 업무에 연결해 설명한다.
4. builder로 그래프를 설계하고 `compile()`한 뒤 `invoke()`로 실행한다.
5. Node가 반환한 **부분 State update**가 기존 State와 합쳐지는 과정을 추적한다.
6. 규칙 기반 분기 함수와 **route label → node name mapping**을 이용해 조건부 흐름을 만든다.

## 학습 방법: 예측 → 실행 → 설명 → 빈칸 → 전이

이 노트북은 다음 순서를 반복합니다.

```text
① Predict  실행 전에 State와 다음 Node를 예상한다.
② Run      코드를 실행해 예상과 비교한다.
③ Explain  표를 이용해 값과 경로가 달라진 이유를 설명한다.
④ Fill     핵심 코드 2~4곳을 직접 완성한다.
⑤ Transfer 같은 구조를 새로운 업무 규칙에 적용한다.
```

`<details>` 안에는 정답이 있습니다. 먼저 종이나 메모장에 답을 적은 뒤 펼쳐 보세요. 실습용 코드 셀은 `Run All`이 멈추지 않도록 주석 처리되어 있습니다. 실습할 때만 복사하거나 주석을 해제합니다.

## 하나의 사례로 끝까지 배우기: 대학 학생지원 문의

학생지원센터에는 다음과 같은 문의가 들어옵니다.

- “수강신청 정정 기간이 언제인가요?”
- “전공 학점 인정 기준이 궁금해요.”
- “학생증을 잃어버렸어요.”
- “장학금 신청 방법을 알고 싶어요.”

처음에는 모든 문의를 똑같이 접수하는 선형 workflow를 만듭니다. 그다음 질문 내용을 규칙으로 분류하여 **학사 문의**와 **일반 문의**를 서로 다른 담당 흐름으로 보냅니다. 마지막 전이 과제에서는 **장학 문의** 경로를 직접 추가합니다.

이렇게 한 사례를 계속 확장하면 새 문법보다 State와 흐름의 변화에 집중할 수 있습니다.

# 1. Agentic AI를 이해하는 mental model

Agentic AI를 이해하려면 먼저 “어떤 기술을 썼는가?”보다 **누가 다음 행동을 결정하는가?**를 물어야 합니다.

```text
사용자 요청
   ↓
[정해진 규칙이 결정] ─────────────→ Workflow
   ↓
[모델이 상황을 해석하거나 선택] ─→ Agent가 포함된 시스템
```

- **LLM**은 문장을 입력받아 다음에 올 법한 문장을 생성하는 모델입니다. 모델 하나가 업무 순서, 데이터 저장, 외부 행동을 자동으로 모두 관리하는 것은 아닙니다.
- **Workflow**는 사람이 미리 정한 단계와 규칙대로 실행되는 절차입니다.
- **Agent**는 주어진 목표와 현재 상황을 바탕으로 모델이 다음 행동이나 사용할 기능을 선택하는 구성요소입니다.
- **Agentic AI 시스템**은 보통 이 둘을 섞습니다. 위험한 업무 규칙은 코드로 고정하고, 해석이 필요한 일부 판단만 모델에 맡깁니다.

> LangGraph를 사용했다고 모든 그래프가 agent인 것은 아닙니다. 이번 주 그래프는 의도적으로 모델이 없는 workflow입니다. 제어 구조를 먼저 이해해야 나중에 모델의 선택이 들어와도 어디까지가 규칙이고 어디부터가 불확실한 판단인지 구분할 수 있습니다.

### LLM, workflow, agent 비교

| 구분 | 다음 행동을 주로 결정하는 주체 | 같은 입력의 결과 | 학생지원 예시 | 주의점 |
|---|---|---|---|---|
| LLM 호출 | 모델 | 달라질 수 있음 | 문의 문장을 자연어로 요약 | 사실 오류와 표현 변동 가능 |
| 규칙 기반 workflow | 사람이 작성한 코드 | 보통 같음 | `수강`이 있으면 학사 담당으로 전달 | 규칙에 없는 표현을 놓칠 수 있음 |
| agent 포함 workflow | 모델과 코드가 역할 분담 | 일부 달라질 수 있음 | 모델이 필요한 업무 기능을 선택하고 코드는 권한을 검사 | 종료·비용·권한 통제가 필요 |

경영 관점에서 핵심은 “agent가 더 똑똑하다”가 아닙니다. **예측 가능성, 유연성, 비용, 통제 가능성** 사이에서 어떤 설계를 선택할지 판단하는 것입니다.

### 이해 확인: 누가 결정하는가?

다음 세 상황을 LLM, workflow, agent 관점에서 분류해 보세요.

1. `문의에 '수강'이 있으면 학사팀으로 보낸다`라는 코드가 경로를 정한다.
2. 모델에게 문의를 한 문장으로 요약해 달라고 하고 요약만 받는다.
3. 모델이 현재 문의를 보고 학사 규정 조회와 상담원 연결 중 다음 행동을 선택한다.

<details><summary>정답과 이유</summary>

1. **규칙 기반 workflow**입니다. 사람이 작성한 조건문이 다음 경로를 정합니다.
2. **LLM 호출**입니다. 모델이 텍스트를 만들지만 다음 업무 행동을 선택하지는 않습니다.
3. **agent가 포함된 workflow**입니다. 모델의 선택이 다음 행동에 영향을 줍니다.

중요한 것은 제품 이름이 아니라 의사결정권의 위치입니다.
</details>

## 1-1. 평범한 `if`/`while`과 그래프는 경쟁 관계가 아니다

작고 고정된 절차는 일반 파이썬 함수가 가장 단순합니다. 아래처럼 단계가 두세 개뿐이라면 굳이 그래프가 필요하지 않을 수 있습니다.

그래프가 유용해지는 시점은 다음과 같습니다.

- 처리 단계와 분기가 많아 전체 흐름을 그림으로 보고 싶을 때
- 여러 단계가 같은 업무 데이터를 공유해야 할 때
- 중간 상태를 저장·검사하거나 특정 단계부터 다시 시작해야 할 때
- 사람의 승인, 실패 재시도, 반복 같은 흐름이 추가될 때
- 각 단계를 독립적으로 테스트하고 담당자를 나누고 싶을 때

> 그래프는 일반 파이썬을 대체하지 않습니다. Node도 결국 파이썬 함수이고, 조건 분기도 결국 파이썬 규칙입니다. 그래프는 이 함수들의 **실행 구조와 공유 상태를 명시적으로 보이게 하는 도구**입니다.

### 먼저 일반 파이썬으로 같은 업무를 표현해 보기

아래 함수는 질문의 공백을 정리하고, 학사 관련 키워드가 있으면 학사 문의로 분류합니다. 작은 문제에서는 이 코드가 충분히 읽기 쉽습니다.

In [ ]:
def plain_support_workflow(student_name, question):
    cleaned = question.strip()
    if "수강" in cleaned or "학점" in cleaned or "전공" in cleaned:
        category = "academic"
        response = f"{student_name}님, 학사 문의로 접수했습니다."
    else:
        category = "general"
        response = f"{student_name}님, 일반 문의로 접수했습니다."

    return {
        "student_name": student_name,
        "question": question,
        "cleaned_question": cleaned,
        "category": category,
        "response": response,
    }

plain_result = plain_support_workflow("민지", "  수강신청 정정 기간이 언제인가요?  ")
print(plain_result)
assert plain_result["category"] == "academic"

### 실행 결과 해석

일반 함수에서는 정리, 분류, 답변 생성, 결과 조립이 한 함수 안에 있습니다. 현재 규모에서는 이것이 단순하고 좋은 선택입니다.

하지만 규칙이 2개에서 20개로 늘고, 문의 종류마다 재시도·승인·기록 단계가 달라진다면 한 함수 안의 `if/while`을 따라가기 어려워집니다. 그래프에서는 다음처럼 책임을 나눌 수 있습니다.

```text
[질문 정리 함수] → [분류 함수] → 조건에 따라 [학사 답변] 또는 [일반 답변]
```

각 상자는 독립된 함수이고 화살표는 실행 관계입니다. LangGraph는 이 구조를 코드로 등록하고 실행합니다.

# 2. 그래프를 만들기 전에 State부터 이해하기

학생지원 workflow의 여러 단계는 같은 문의 정보를 함께 사용합니다. 한 단계는 질문을 정리하고, 다음 단계는 분류 결과를 추가하며, 마지막 단계는 답변을 만듭니다.

이때 단계 사이를 이동하는 공용 업무 기록을 **State**라고 부릅니다.

**업무 비유**

- State: 문의 한 건의 전자 업무 카드
- Node: 카드의 내용을 읽고 자기 담당 칸을 작성하는 직원
- Edge: 카드가 다음 직원에게 이동하는 경로

State는 그냥 “변수 하나”라기보다, 그래프가 실행되는 동안 공유되는 **이름 붙은 데이터 묶음**이라고 생각하면 좋습니다.

## 2-1. 필요한 파이썬 문법 1분 복습

Node는 State 딕셔너리를 입력받아, 새로 쓰거나 바꿀 값만 딕셔너리로 반환합니다. 아래 코드는 아직 LangGraph가 아니라 이 패턴을 확인하는 순수 파이썬입니다.

In [ ]:
sample_state = {
    "student_name": "민지",
    "question": "  수강신청 기간이 궁금해요.  ",
}

def make_clean_update(state):
    return {"cleaned_question": state["question"].strip()}

sample_update = make_clean_update(sample_state)
print("원래 State:", sample_state)
print("함수가 반환한 update:", sample_update)
assert "cleaned_question" not in sample_state

### 원본 State와 update를 구분하기

| 항목 | 들어 있는 값 | 의미 |
|---|---|---|
| `sample_state` | 이름, 원래 질문 | 함수가 읽은 현재 업무 카드 |
| `sample_update` | 정리된 질문만 | 이 함수가 새로 작성할 칸 |

`make_clean_update()`는 원본 딕셔너리를 직접 바꾸지 않았습니다. LangGraph는 Node의 반환값을 받아 기존 State에 같은 이름의 칸은 덮어쓰고, 새로운 이름의 칸은 추가합니다. 반환하지 않은 기존 칸은 유지됩니다.

```text
기존 State                         Node가 반환한 update
{student_name, question}     +     {cleaned_question}
                              ↓ LangGraph가 key별로 합침
다음 State
{student_name, question, cleaned_question}
```

## 2-2. `TypedDict`는 State의 설계 설명서

딕셔너리는 자유롭지만, 키 이름을 잘못 쓰면 오류가 실행할 때까지 드러나지 않을 수 있습니다. `TypedDict`로 “이 State에는 어떤 칸과 자료형을 사용할 것인지” 문서화할 수 있습니다.

- `student_name: str`: 학생 이름은 문자열이어야 한다는 타입 힌트
- `NotRequired[str]`: 이 문자열 칸은 초기 입력에 아직 없어도 된다는 타입 힌트

중요한 사실은 두 가지입니다.

1. 타입 힌트는 값을 자동으로 만들지 않습니다.
2. `TypedDict`는 보통 실행 중 값을 검사하거나 잘못된 타입을 자동 수정하지 않습니다.

즉, `response: NotRequired[str]`라고 적었다고 빈 답변이 생기는 것이 아닙니다. 실제 값은 Node가 반환해야 생깁니다.

In [ ]:
from typing import Literal, NotRequired, TypedDict
from langgraph.graph import END, START, StateGraph

class InquiryState(TypedDict):
    student_name: str
    question: str
    cleaned_question: NotRequired[str]
    category: NotRequired[str]
    response: NotRequired[str]

print("✅ InquiryState 설계 설명서를 정의했습니다.")

### 초기 State에는 아직 결과 칸이 없다

처음 접수한 순간에는 학생 이름과 질문만 있습니다. 정리된 질문, 분류, 답변은 뒤의 Node가 차례로 만듭니다.

In [ ]:
initial_inquiry: InquiryState = {
    "student_name": "민지",
    "question": "  수강신청 정정 기간이 언제인가요?  ",
}

print("초기 State:", initial_inquiry)
assert "cleaned_question" not in initial_inquiry
assert "response" not in initial_inquiry

### 흔한 오해: 스키마와 실제 값은 다르다

| 오해 | 실제 |
|---|---|
| `InquiryState`를 정의하면 State가 하나 생성된다. | 클래스 형태의 설계 설명서만 정의된다. |
| `NotRequired` 칸에는 자동으로 `None`이 들어간다. | 키 자체가 존재하지 않는다. |
| 타입 힌트가 잘못된 값을 항상 막아 준다. | 주로 코드 편집기와 사람이 코드를 읽도록 돕는다. |
| 모든 Node가 모든 키를 반환해야 한다. | 자기 단계가 갱신할 일부 키만 반환하면 된다. |

실제 코드에서 없는 키를 `state["response"]`처럼 너무 일찍 읽으면 `KeyError`가 납니다. 따라서 **어떤 Node가 어떤 키를 먼저 만드는지**가 Edge 순서만큼 중요합니다.

## 2-3. State, Node, Edge, START, END

| 개념 | 코드에서의 모습 | 학생지원 업무 비유 |
|---|---|---|
| State | `InquiryState` 형태의 딕셔너리 | 문의 업무 카드 |
| Node | `def ... (state):` 함수 | 한 가지 일을 담당하는 직원 |
| Edge | `add_edge(A, B)` | 카드가 A에서 B로 가는 전달선 |
| `START` | 실제 함수가 없는 가상 시작점 | 문의가 접수함에 들어온 순간 |
| `END` | 실제 함수가 없는 가상 종료점 | 이번 처리 흐름이 끝난 순간 |

`START`와 `END`는 직접 실행하는 Node가 아닙니다. 그래프의 입구와 출구를 표시하는 특별한 표지입니다.

## 2-4. build, compile, invoke를 구분하기

LangGraph 작업은 크게 세 단계입니다.

| 단계 | 대표 코드 | 결과 |
|---|---|---|
| Build: 설계 | `StateGraph`, `add_node`, `add_edge` | 수정 가능한 builder |
| Compile: 실행 준비 | `graph = builder.compile()` | 실행 가능한 compiled graph |
| Invoke: 한 번 실행 | `graph.invoke(initial_state)` | `START`부터 `END`까지 실행한 최종 State |

여기서 **build는 개념적인 단계 이름**입니다. 이 수업에서 `build()`라는 메서드를 호출하지 않습니다. `StateGraph(...)`로 builder를 만들고 Node와 Edge를 등록하는 과정 전체를 build라고 부릅니다.

```text
builder(설계도) -- compile() --> compiled graph(작동 가능한 업무 시스템)
compiled graph + 초기 State -- invoke() --> 최종 State
```

`invoke()`는 Node 하나를 실행하는 버튼이 아니라, 초기 State를 넣어 **그래프 전체를 끝까지 한 번 실행**하는 버튼입니다. compile 뒤에 builder를 바꾸더라도 이미 만든 compiled graph가 자동으로 바뀌지는 않으므로 다시 compile해야 합니다.

### 개념 점검

1. Node 함수가 `{"category": "academic"}`만 반환하면 기존 `student_name`은 사라질까요?
2. `START`는 우리가 정의해야 하는 파이썬 함수일까요?
3. `linear_builder.invoke(...)`와 `linear_graph.invoke(...)` 중 어느 쪽을 실행할까요?

<details><summary>정답 보기</summary>

1. 사라지지 않습니다. 반환하지 않은 기존 키는 유지됩니다.
2. 아닙니다. 그래프의 가상 입구를 나타내는 특별한 값입니다.
3. `compile()`이 반환한 `linear_graph`를 실행합니다. builder는 설계도입니다.
</details>

# 3. 첫 번째 그래프: 모든 문의를 같은 순서로 처리하기

첫 버전은 분기가 없습니다. 질문의 앞뒤 공백을 정리한 뒤 접수 답변을 만듭니다.

```text
START → normalize_question → draft_reception_reply → END
```

각 Node는 한 가지 책임만 갖습니다.

- `normalize_question`: 원문을 읽고 `cleaned_question`을 만든다.
- `draft_reception_reply`: 이름과 정리된 질문을 읽고 `response`를 만든다.

## 3-1. Node 함수 정의

Node 함수는 현재 State를 받습니다. 원본을 직접 수정하는 대신 이번 단계가 바꿀 **부분 update 딕셔너리**를 반환합니다.

In [ ]:
def normalize_question(state: InquiryState) -> dict:
    cleaned = state["question"].strip()
    return {"cleaned_question": cleaned}

def draft_reception_reply(state: InquiryState) -> dict:
    name = state["student_name"]
    question = state["cleaned_question"]
    response = f"{name}님, '{question}' 문의가 접수되었습니다."
    return {"response": response}

print("✅ 두 Node 함수를 정의했습니다.")

### Node가 읽는 키와 쓰는 키

| Node | 읽는 키 | 반환하여 쓰는 키 |
|---|---|---|
| `normalize_question` | `question` | `cleaned_question` |
| `draft_reception_reply` | `student_name`, `cleaned_question` | `response` |

`draft_reception_reply`가 `cleaned_question`을 읽으므로 반드시 `normalize_question` 뒤에 실행되어야 합니다. Edge 순서를 반대로 연결하면 아직 없는 키를 읽게 됩니다.

## 3-2. builder에 Node와 Edge 등록하기

함수를 정의한 것만으로는 실행 순서가 생기지 않습니다. 어떤 이름으로 Node를 등록할지, 어떤 순서로 연결할지를 builder에 적습니다.

Node 이름 문자열과 파이썬 함수 이름은 같게 쓰면 읽기 쉽지만 반드시 같아야 하는 것은 아닙니다.

In [ ]:
linear_builder = StateGraph(InquiryState)

linear_builder.add_node("normalize_question", normalize_question)
linear_builder.add_node("draft_reception_reply", draft_reception_reply)

linear_builder.add_edge(START, "normalize_question")
linear_builder.add_edge("normalize_question", "draft_reception_reply")
linear_builder.add_edge("draft_reception_reply", END)

linear_graph = linear_builder.compile()
print("✅ 선형 그래프를 compile했습니다.")

### builder와 compiled graph 확인

- `linear_builder`에는 State 설계, Node 목록, Edge가 담겨 있습니다.
- `linear_graph`는 `invoke()`할 수 있는 실행 객체입니다.
- `compile()`은 Node 함수를 실행하지 않습니다. 실행 가능한 구조를 준비할 뿐입니다.

비유하면 builder는 수정 중인 업무 절차도이고, compiled graph는 실제로 문의 카드를 흘려보낼 수 있는 운영 시스템입니다.

## 3-3. Predict: 실행 전에 State를 그려 보기

초기 입력은 다음과 같습니다.

```python
{
    "student_name": "민지",
    "question": "  수강신청 정정 기간이 언제인가요?  ",
}
```

코드를 실행하기 전에 답해 보세요.

1. `normalize_question` 뒤에 새로 생기는 키는 무엇인가요?
2. `question`의 앞뒤 공백은 원래 키에서 없어질까요, 새 키에서만 없어질까요?
3. `draft_reception_reply`가 실행되기 직전에는 어떤 키가 세 개 존재할까요?
4. 최종 State에는 입력의 `question`도 남아 있을까요?

In [ ]:
linear_input: InquiryState = {
    "student_name": "민지",
    "question": "  수강신청 정정 기간이 언제인가요?  ",
}

linear_result = linear_graph.invoke(linear_input)

print("초기 입력:", linear_input)
print("최종 State:", linear_result)

assert linear_result["cleaned_question"] == "수강신청 정정 기간이 언제인가요?"
assert linear_result["response"].endswith("문의가 접수되었습니다.")
assert "cleaned_question" not in linear_input
print("✅ 예상한 State 변화가 확인되었습니다.")

## 3-4. Explain: 실행 결과를 단계별로 읽기

| 시점 | `student_name` | `question` | `cleaned_question` | `response` |
|---|---|---|---|---|
| 초기 입력 | 민지 | 앞뒤 공백이 있는 원문 | 없음 | 없음 |
| `normalize_question` 후 | 민지 | 원문 유지 | 공백을 제거한 문장 | 없음 |
| `draft_reception_reply` 후 | 민지 | 원문 유지 | 그대로 유지 | 접수 답변 생성 |

핵심은 **Node 반환값이 최종 State 전체가 아니라 update**라는 점입니다.

```text
`normalize_question` 반환: {cleaned_question}
LangGraph가 만든 다음 State: {student_name, question, cleaned_question}
```

원래 `question`을 반환하지 않았는데도 남아 있는 이유는 LangGraph가 key별 update를 기존 State에 합쳤기 때문입니다. 같은 키를 반환하면 그 키의 이전 값이 새 값으로 바뀌고, 반환하지 않은 키는 유지됩니다.

### 선형 그래프에서 자주 하는 실수

| 실수 | 나타나는 현상 | 확인할 것 |
|---|---|---|
| `draft_reception_reply`를 먼저 연결 | `cleaned_question` `KeyError` | 그 키를 만드는 Node가 먼저인지 |
| Node가 문자열만 반환 | State update 형식 오류 | `{"response": ...}` 딕셔너리인지 |
| builder를 `invoke()` | 실행 메서드 오류 | compile 결과 객체를 실행했는지 |
| builder 수정 후 예전 graph 실행 | 새 Node가 실행되지 않음 | 수정 뒤 다시 compile했는지 |
| Node 이름 문자열 오타 | compile 또는 실행 중 경로 오류 | 등록 이름과 Edge 이름이 같은지 |

## 3-5. Fill: 안내 문구 Node 추가하기

`draft_reception_reply` 뒤에 `add_status_note` Node를 추가해 답변 끝에 ` 현재 상태: 접수 완료.`를 붙이세요.

아래 코드에는 정확히 네 개의 `____`가 있습니다. 빈칸을 채운 뒤 주석을 해제하여 실행하세요.

성공 기준:

- `START → normalize_question → draft_reception_reply → add_status_note → END` 순서
- 최종 `response`가 `현재 상태: 접수 완료.`로 끝남
- 기존 `student_name`, `question`, `cleaned_question` 유지

In [ ]:
# def add_status_note(state: InquiryState) -> dict:
#     return {"response": ____ + " 현재 상태: 접수 완료."}  # ①
#
# practice_builder = StateGraph(InquiryState)
# practice_builder.add_node("normalize_question", normalize_question)
# practice_builder.add_node("draft_reception_reply", draft_reception_reply)
# practice_builder.add_node(____, add_status_note)  # ②
# practice_builder.add_edge(START, "normalize_question")
# practice_builder.add_edge("normalize_question", "draft_reception_reply")
# practice_builder.add_edge(____, "add_status_note")  # ③
# practice_builder.add_edge(____, END)  # ④
# practice_graph = practice_builder.compile()
#
# practice_result = practice_graph.invoke(linear_input)
# print(practice_result["response"])
# assert practice_result["response"].endswith("현재 상태: 접수 완료.")

<details><summary>빈칸 정답과 완성 코드</summary>

① `state["response"]`  
② `"add_status_note"`  
③ `"draft_reception_reply"`  
④ `"add_status_note"`

```python
def add_status_note(state: InquiryState) -> dict:
    return {"response": state["response"] + " 현재 상태: 접수 완료."}

practice_builder = StateGraph(InquiryState)
practice_builder.add_node("normalize_question", normalize_question)
practice_builder.add_node("draft_reception_reply", draft_reception_reply)
practice_builder.add_node("add_status_note", add_status_note)
practice_builder.add_edge(START, "normalize_question")
practice_builder.add_edge("normalize_question", "draft_reception_reply")
practice_builder.add_edge("draft_reception_reply", "add_status_note")
practice_builder.add_edge("add_status_note", END)
practice_graph = practice_builder.compile()

practice_result = practice_graph.invoke(linear_input)
assert practice_result["response"].endswith("현재 상태: 접수 완료.")
```

이 실습에서도 `add_status_note`는 `response`만 반환합니다. 다른 키를 반환하지 않아도 기존 값은 유지됩니다.
</details>

# 4. 조건부 Edge: 문의 내용에 따라 다른 경로로 보내기

모든 문의를 같은 문장으로 접수하는 첫 그래프는 업무 규칙을 충분히 반영하지 못합니다. 이제 질문에 학사 관련 키워드가 있으면 학사 안내를, 그렇지 않으면 일반 안내를 만들겠습니다.

```text
START → normalize_question → classify_inquiry
                                ├─ academic_route → answer_academic → END
                                └─ general_route  → answer_general  → END
```

이번 분기는 모델의 판단이 아니라 우리가 쓴 문자열 규칙으로 결정됩니다. 따라서 같은 입력은 같은 경로로 갑니다.

## 4-1. 작업 Node와 router 함수는 역할이 다르다

| 구성요소 | 무엇을 반환하는가 | State를 바꾸는가? |
|---|---|---|
| `classify_inquiry` Node | `{"category": ...}` update | 예 |
| `select_route` router | `"academic_route"` 같은 route label | 아니요 |
| 답변 Node | `{"response": ...}` update | 예 |

router가 반환하는 문자열은 State 데이터가 아니라 **Edge가 어느 길을 선택할지 알려 주는 표지**입니다. route label을 State에도 남기고 싶다면 별도의 Node가 그 값을 update로 반환해야 합니다.

## 4-2. 분류 Node와 router 정의

`classify_inquiry`는 정리된 질문을 읽어 `category`를 만듭니다. 그다음 `select_route`는 category를 보고 route label을 반환합니다.

`Literal[...]`은 가능한 반환 문자열을 사람과 코드 편집기에게 알려 주는 타입 힌트입니다. 실행 중 잘못된 label을 자동으로 올바르게 고쳐 주지는 않습니다.

In [ ]:
def classify_inquiry(state: InquiryState) -> dict:
    text = state["cleaned_question"]
    if "수강" in text or "학점" in text or "전공" in text:
        category = "academic"
    else:
        category = "general"
    return {"category": category}

def select_route(
    state: InquiryState,
) -> Literal["academic_route", "general_route"]:
    if state["category"] == "academic":
        return "academic_route"
    return "general_route"

## 4-3. 각 경로의 답변 Node 정의

두 답변 Node는 모두 같은 State를 읽지만 서로 다른 업무 문장을 만듭니다. route label은 아직 이 함수들의 이름과 다릅니다.

- route label: `academic_route`, `general_route`
- node name: `answer_academic`, `answer_general`

다음 단계에서 둘 사이의 mapping을 명시합니다.

In [ ]:
def answer_academic(state: InquiryState) -> dict:
    name = state["student_name"]
    response = f"{name}님, 학사 문의로 분류했습니다. 학사 공지를 확인해 주세요."
    return {"response": response}

def answer_general(state: InquiryState) -> dict:
    name = state["student_name"]
    response = f"{name}님, 일반 학생지원 문의로 분류했습니다."
    return {"response": response}

print("✅ 두 경로의 답변 Node를 정의했습니다.")

## 4-4. route label과 node name을 mapping하기

router가 `academic_route`를 반환해도 그 이름의 Node를 자동으로 찾는 것은 아닙니다. 아래 딕셔너리의 왼쪽은 router의 반환 label, 오른쪽은 실제 등록된 node name입니다.

```python
{
    "academic_route": "answer_academic",
    "general_route": "answer_general",
}
```

label과 node name을 일부러 다르게 두면 두 역할을 분명하게 볼 수 있습니다. 실제 프로젝트에서도 업무 의미가 담긴 label을 내부 함수 이름과 독립적으로 관리할 수 있습니다.

In [ ]:
route_map = {
    "academic_route": "answer_academic",
    "general_route": "answer_general",
}

support_builder = StateGraph(InquiryState)
support_builder.add_node("normalize_question", normalize_question)
support_builder.add_node("classify_inquiry", classify_inquiry)
support_builder.add_node("answer_academic", answer_academic)
support_builder.add_node("answer_general", answer_general)

support_builder.add_edge(START, "normalize_question")
support_builder.add_edge("normalize_question", "classify_inquiry")
support_builder.add_conditional_edges("classify_inquiry", select_route, route_map)
support_builder.add_edge("answer_academic", END)
support_builder.add_edge("answer_general", END)

support_graph = support_builder.compile()
print("✅ 조건 분기 그래프를 compile했습니다.")

## 4-5. Predict: 두 문의의 경로 예상하기

아래 두 입력을 실행하기 전에 표의 빈칸을 채워 보세요.

| 질문 | 예상 `category` | router label | 실행될 답변 Node |
|---|---|---|---|
| `전공 학점 인정 기준이 궁금해요.` | ? | ? | ? |
| `학생증을 잃어버렸어요.` | ? | ? | ? |

추가 질문:

1. 최종 State 안에 `academic_route`라는 값이 자동으로 저장될까요?
2. 두 경로의 답변 Node가 모두 실행될까요?
3. `invoke()`는 분류 Node에서 멈출까요, 선택된 답변 Node와 `END`까지 갈까요?

In [ ]:
test_inquiries = [
    {"student_name": "준호", "question": "전공 학점 인정 기준이 궁금해요."},
    {"student_name": "서연", "question": "학생증을 잃어버렸어요."},
]

routing_results = []
for inquiry in test_inquiries:
    result = support_graph.invoke(inquiry)
    routing_results.append(result)
    print("질문:", result["cleaned_question"])
    print("분류:", result["category"])
    print("답변:", result["response"])
    print("-" * 40)

assert routing_results[0]["category"] == "academic"
assert routing_results[1]["category"] == "general"
print("✅ 두 입력이 서로 다른 경로를 통과했습니다.")

## 4-6. Explain: 조건 분기 실행 결과

| 시점 | 첫 번째 문의 State 변화 | 두 번째 문의 State 변화 |
|---|---|---|
| 초기 | 이름 + 전공 학점 질문 | 이름 + 학생증 질문 |
| `normalize_question` 후 | `cleaned_question` 추가 | `cleaned_question` 추가 |
| `classify_inquiry` 후 | `category="academic"` 추가 | `category="general"` 추가 |
| `select_route` | `academic_route` label 반환, State 변화 없음 | `general_route` label 반환, State 변화 없음 |
| mapping 결과 | `answer_academic` 실행 | `answer_general` 실행 |
| 최종 | 학사 안내 `response` 추가 | 일반 안내 `response` 추가 |

`select_route`가 반환한 label은 경로 선택에 소비되므로 최종 State에 자동 저장되지 않습니다. 또한 조건부 Edge는 두 답변 Node를 모두 실행하는 것이 아니라 입력에 맞는 한 경로만 선택합니다.

### 세 가지 결정 방식을 연결해서 보기

| 그래프 위치 | 다음 단계를 결정한 것 | 결정 방식 |
|---|---|---|
| `START → normalize_question` | 개발자가 등록한 고정 Edge | 항상 동일 |
| `classify_inquiry` 내부 | 개발자가 작성한 `if` 규칙 | 입력값에 따라 결정론적 |
| `select_route` 이후 | router label과 `route_map` | 선택한 한 Node로 이동 |

조건부 그래프라고 해서 곧바로 agent가 되는 것은 아닙니다. 현재 다음 행동은 모두 사람이 작성한 규칙이 결정합니다.

### 조건부 Edge에서 자주 하는 실수

| 실수 | 결과 | 해결 |
|---|---|---|
| router가 `{"route": ...}` 딕셔너리를 반환 | Edge가 label을 찾지 못함 | router는 label 문자열을 반환 |
| `route_map` 왼쪽에 node name을 적음 | router 반환 label과 불일치 | 왼쪽은 label, 오른쪽은 node name |
| 새로운 Node만 등록하고 mapping에는 누락 | 해당 경로로 이동 불가 | Node·label·mapping을 함께 추가 |
| `category`를 만들기 전에 router 실행 | `KeyError` | 분류 Node 뒤에 조건부 Edge 연결 |
| router label 철자 불일치 | 실행할 경로를 찾지 못함 | `Literal`, mapping, 반환 문자열 비교 |

## 4-7. Fill: 조건 그래프를 다시 조립하기

아래에는 네 개의 빈칸이 있습니다. 기존에 정의한 함수와 `route_map`을 사용해 같은 구조를 다시 compile하세요. 빈칸을 채우고 주석을 해제한 뒤 마지막 assert까지 실행합니다.

In [ ]:
# review_builder = StateGraph(____)  # ① State schema
# review_builder.add_node("normalize_question", ____)  # ② 함수
# review_builder.add_node("classify_inquiry", classify_inquiry)
# review_builder.add_node("answer_academic", answer_academic)
# review_builder.add_node("answer_general", answer_general)
#
# review_builder.add_edge(START, "normalize_question")
# review_builder.add_edge("normalize_question", "classify_inquiry")
# review_builder.add_conditional_edges(
#     "classify_inquiry", select_route, ____  # ③ label mapping
# )
# review_builder.add_edge("answer_academic", END)
# review_builder.add_edge("answer_general", END)
#
# review_graph = ____.compile()  # ④ builder
# review_result = review_graph.invoke(test_inquiries[0])
# assert review_result["category"] == "academic"
# print("✅ 조건 그래프 재조립 성공")

<details><summary>빈칸 정답</summary>

① `InquiryState`  
② `normalize_question`  
③ `route_map`  
④ `review_builder`

이 실습의 핵심은 함수 정의와 그래프 조립이 별개라는 점입니다. 같은 Node 함수도 다른 builder에서 다른 Edge 구조로 재사용할 수 있습니다.
</details>

# 5. Transfer: 장학 문의 경로를 직접 추가하기

이제 따라 만들기에서 벗어나 업무 규칙을 확장합니다. 다음 요구사항을 만족하는 `support_graph_v2`를 만드세요.

## 업무 요구사항

1. 질문에 `장학` 또는 `등록금`이 들어 있으면 `category="scholarship"`으로 분류한다.
2. router는 이 경우 `scholarship_route` label을 반환한다.
3. mapping은 그 label을 `answer_scholarship` Node에 연결한다.
4. 장학 답변은 학생 이름을 포함하고 “장학 공지”를 확인하라고 안내한다.
5. 기존 academic/general 경로도 그대로 작동해야 한다.

```text
                        ┌─ academic_route    → answer_academic
classify_inquiry_v2 ────┼─ scholarship_route → answer_scholarship
                        └─ general_route     → answer_general
```

## 5-1. 구현 전 설계표

코드보다 먼저 아래 표를 완성하세요.

| 입력 질문 | 분류 Node가 추가할 `category` | router label | 실행 Node |
|---|---|---|---|
| 장학금 신청 기간이 궁금해요. | ? | ? | ? |
| 전공 학점 인정 기준이 궁금해요. | ? | ? | ? |
| 학생증을 잃어버렸어요. | ? | ? | ? |

그리고 새 State key가 필요한지 생각해 보세요. 이번 요구사항은 기존 `category`와 `response`를 재사용할 수 있으므로 스키마를 바꿀 필요가 없습니다.

## 5-2. Scaffold A: 분류·router·답변 함수

아래 주석을 해제하고 `TODO`를 완성하세요. 기존 함수 이름과 충돌하지 않도록 `_v2`를 붙였습니다.

In [ ]:
# def classify_inquiry_v2(state: InquiryState) -> dict:
#     text = state["cleaned_question"]
#     # TODO: scholarship, academic, general 순으로 category 결정
#     category = "TODO"
#     return {"category": category}
#
# def select_route_v2(state: InquiryState):
#     # TODO: category에 맞는 세 route label 중 하나 반환
#     return "TODO"
#
# def answer_scholarship(state: InquiryState) -> dict:
#     # TODO: 이름과 '장학 공지'를 포함한 response 반환
#     return {}

## 5-3. Scaffold B: 그래프 조립

Node를 하나 늘릴 때에는 다음 네 곳을 함께 점검합니다.

1. 함수 정의
2. `add_node` 등록
3. router가 반환할 label
4. label에서 node name으로 가는 mapping과 `END` Edge

In [ ]:
# route_map_v2 = {
#     "academic_route": "answer_academic",
#     "scholarship_route": "TODO",
#     "general_route": "answer_general",
# }
#
# support_builder_v2 = StateGraph(InquiryState)
# support_builder_v2.add_node("normalize_question", normalize_question)
# support_builder_v2.add_node("classify_inquiry_v2", classify_inquiry_v2)
# support_builder_v2.add_node("answer_academic", answer_academic)
# support_builder_v2.add_node("answer_scholarship", answer_scholarship)
# support_builder_v2.add_node("answer_general", answer_general)
# support_builder_v2.add_edge(START, "normalize_question")
# support_builder_v2.add_edge("normalize_question", "classify_inquiry_v2")
# support_builder_v2.add_conditional_edges(
#     "classify_inquiry_v2", select_route_v2, route_map_v2
# )
# # TODO: 세 답변 Node를 각각 END에 연결
# support_graph_v2 = support_builder_v2.compile()

## 5-4. Scaffold C: 세 경로를 모두 검사하기

한 예제만 성공하면 다른 경로가 망가진 것을 놓칠 수 있습니다. 세 범주를 모두 테스트하세요.

In [ ]:
# scholarship_case = {
#     "student_name": "하린",
#     "question": "장학금 신청 기간이 궁금해요.",
# }
# academic_case = {
#     "student_name": "준호",
#     "question": "전공 학점 인정 기준이 궁금해요.",
# }
# general_case = {"student_name": "서연", "question": "학생증을 잃어버렸어요."}
#
# assert support_graph_v2.invoke(scholarship_case)["category"] == "scholarship"
# assert support_graph_v2.invoke(academic_case)["category"] == "academic"
# assert support_graph_v2.invoke(general_case)["category"] == "general"
# print("✅ 세 경로 테스트 통과")

## 5-5. 제출물과 자기평가 rubric

제출물:

1. 실행 가능한 `support_graph_v2` 코드
2. 장학·학사·일반 입력의 최종 State 또는 출력 캡처
3. 장학 입력 하나에 대한 Node별 State 변화표
4. “분류 Node, router, mapping의 역할 차이”를 설명하는 세 문장

| 평가 항목 | 2점 | 1점 | 0점 |
|---|---|---|---|
| 세 경로 동작 | 세 assert 모두 통과 | 일부만 통과 | 실행 불가 |
| State update | 필요한 키만 반환하고 기존 키 유지 | 동작하지만 전체 State를 불필요하게 재작성 | State 흐름 설명 불가 |
| routing 구조 | label과 node mapping을 정확히 구분 | 결과는 맞지만 역할 설명이 모호 | 분기 실패 |
| trace 설명 | Node별 전후 State를 정확히 작성 | 일부 단계 누락 | 최종 답만 제시 |
| 코드 명료성 | 이름·주석이 업무 의미를 드러냄 | 읽을 수 있으나 모호한 이름 존재 | 복사 코드로 구조 파악 어려움 |

총 8점 이상이면 이번 주 핵심을 달성한 것으로 봅니다.

<details><summary>통합 과제 예시 답안 A: 함수</summary>

```python
def classify_inquiry_v2(state: InquiryState) -> dict:
    text = state["cleaned_question"]
    if "장학" in text or "등록금" in text:
        category = "scholarship"
    elif "수강" in text or "학점" in text or "전공" in text:
        category = "academic"
    else:
        category = "general"
    return {"category": category}

def select_route_v2(state: InquiryState):
    if state["category"] == "scholarship":
        return "scholarship_route"
    if state["category"] == "academic":
        return "academic_route"
    return "general_route"

def answer_scholarship(state: InquiryState) -> dict:
    name = state["student_name"]
    return {"response": f"{name}님, 장학 문의로 분류했습니다. 장학 공지를 확인해 주세요."}
```

조건문의 순서도 업무 규칙입니다. 더 구체적인 장학 규칙을 먼저 검사하고, 그다음 학사, 마지막에 일반 경로로 보냅니다.
</details>

<details><summary>통합 과제 예시 답안 B: 조립과 검사</summary>

```python
route_map_v2 = {
    "academic_route": "answer_academic",
    "scholarship_route": "answer_scholarship",
    "general_route": "answer_general",
}

support_builder_v2 = StateGraph(InquiryState)
support_builder_v2.add_node("normalize_question", normalize_question)
support_builder_v2.add_node("classify_inquiry_v2", classify_inquiry_v2)
support_builder_v2.add_node("answer_academic", answer_academic)
support_builder_v2.add_node("answer_scholarship", answer_scholarship)
support_builder_v2.add_node("answer_general", answer_general)
support_builder_v2.add_edge(START, "normalize_question")
support_builder_v2.add_edge("normalize_question", "classify_inquiry_v2")
support_builder_v2.add_conditional_edges(
    "classify_inquiry_v2", select_route_v2, route_map_v2
)
support_builder_v2.add_edge("answer_academic", END)
support_builder_v2.add_edge("answer_scholarship", END)
support_builder_v2.add_edge("answer_general", END)
support_graph_v2 = support_builder_v2.compile()

assert support_graph_v2.invoke(scholarship_case)["category"] == "scholarship"
assert support_graph_v2.invoke(academic_case)["category"] == "academic"
assert support_graph_v2.invoke(general_case)["category"] == "general"
```

함수 이름, label 이름은 다른 방식으로 지어도 괜찮습니다. 요구한 세 경로와 State 변화가 명확하면 정답입니다.
</details>

# 6. 오류를 읽는 순서

초보자에게 오류는 실패가 아니라 **현재 State와 그래프 설계를 비교하는 단서**입니다.

1. 오류의 마지막 줄에서 오류 종류와 key 또는 node 이름을 찾습니다.
2. 오류가 난 Node가 어떤 State key를 읽으려 했는지 확인합니다.
3. 그 key를 만드는 앞 Node가 실제로 연결되어 있는지 확인합니다.
4. router 반환 label과 mapping 왼쪽이 정확히 같은지 확인합니다.
5. builder를 수정했다면 다시 `compile()`했는지 확인합니다.
6. 셀 실행 순서가 꼬였다면 커널을 다시 시작하고 위에서부터 실행합니다.

| 오류/증상 | 가장 먼저 의심할 것 |
|---|---|
| `KeyError: 'cleaned_question'` | normalize Node보다 먼저 해당 키를 읽음 |
| 경로를 찾을 수 없음 | router label이 mapping에 없음 |
| 새 Node가 실행되지 않음 | 예전 compiled graph를 실행 중 |
| 결과가 딕셔너리가 아님 | Node가 State update 대신 문자열 등을 반환 |

### 디버깅 사고 연습

router가 `"scholarship_route"`를 반환했지만 mapping이 아래와 같다면 무슨 일이 생길까요?

```python
broken_map = {
    "academic_route": "answer_academic",
    "general_route": "answer_general",
}
```

<details><summary>정답 보기</summary>

`scholarship_route`에 대응하는 오른쪽 node name이 없으므로 그래프가 다음 Node를 선택할 수 없습니다. `"scholarship_route": "answer_scholarship"` 항목을 추가하고, `answer_scholarship` Node 등록과 `END` Edge도 확인해야 합니다.
</details>

# 7. 오늘의 개념 지도

```text
업무 데이터
   InquiryState
       ↓ 현재 값을 읽음
Node 함수 ── 부분 update 반환 ──→ 다음 State
       ↓
Edge가 다음 Node로 이동

START → [고정 Edge] → Node → [조건부 Edge]
                              router가 route label 반환
                                      ↓
                              mapping이 실제 Node 선택 → END

builder에서 Node·Edge를 설계
          ↓ compile()
compiled graph
          ↓ invoke(initial_state)
최종 State
```

이번 주 그래프는 규칙 기반 workflow입니다. 그래프 구조와 State 변화를 먼저 이해했기 때문에, 이후 더 유연한 판단 요소가 들어와도 시스템의 통제 경계를 구분할 수 있습니다.

## 핵심 용어집

| 용어 | 이 수업에서의 뜻 |
|---|---|
| LLM | 입력 문맥을 바탕으로 텍스트를 생성하는 모델 |
| Workflow | 사람이 정한 단계와 규칙에 따라 실행되는 절차 |
| Agent | 목표와 상황을 보고 모델이 다음 행동을 선택하는 구성요소 |
| Agentic AI system | 모델의 선택과 규칙 기반 통제를 조합한 시스템 |
| Graph | Node와 Edge로 업무 흐름을 표현한 구조 |
| State | 그래프 실행 중 단계들이 공유하는 업무 데이터 |
| State schema | 사용할 key와 예상 타입을 설명하는 설계 |
| `TypedDict` | 딕셔너리 key·타입을 문서화하는 타입 힌트 도구 |
| Node | State를 읽고 부분 update를 반환하는 함수 |
| Partial update | Node가 이번 단계에서 새로 쓰거나 바꿀 key만 담은 딕셔너리 |
| Edge | 한 Node 다음에 실행할 Node를 연결한 경로 |
| `START` / `END` | 그래프의 가상 입구와 출구 |
| Builder | Node와 Edge를 등록하는 수정 가능한 설계도 |
| `compile()` | builder를 실행 가능한 graph 객체로 만드는 단계 |
| `invoke()` | 초기 State로 그래프 전체를 한 번 실행하는 메서드 |
| Router | 현재 State를 읽고 다음 경로 label을 반환하는 함수 |
| Route label | router가 선택한 경로를 나타내는 문자열 |
| Path mapping | route label을 실제 node name과 연결하는 딕셔너리 |
| Deterministic | 같은 입력과 규칙이면 같은 결과가 나오는 성질 |

## 숙달 체크리스트

자료를 닫기 전에 코드 없이 설명해 보세요.

- [ ] LLM 호출, 규칙 workflow, agent가 포함된 workflow를 “누가 다음 행동을 정하는가”로 구분할 수 있다.
- [ ] 작은 `if` 함수가 그래프보다 더 적절할 수 있는 이유를 말할 수 있다.
- [ ] State·Node·Edge를 학생지원 업무 비유와 코드 양쪽으로 설명할 수 있다.
- [ ] `TypedDict`가 값을 만들거나 런타임 검증을 보장하지 않는다고 설명할 수 있다.
- [ ] Node의 부분 update가 기존 State와 어떻게 합쳐지는지 표로 추적할 수 있다.
- [ ] `START`, `END`, builder, compiled graph, `invoke()`를 구분할 수 있다.
- [ ] 선형 그래프를 빈 화면에서 다시 만들 수 있다.
- [ ] 분류 Node와 router의 차이를 설명할 수 있다.
- [ ] route label과 node name이 달라도 mapping으로 연결할 수 있다.
- [ ] 새 경로를 추가할 때 함수·Node 등록·label·mapping·END Edge를 함께 점검할 수 있다.
- [ ] 장학·학사·일반 세 입력에 대한 테스트를 작성할 수 있다.

## Exit ticket

다음 질문에 각각 한두 문장으로 답하면 Week 1을 마친 것입니다.

1. `normalize_question`이 `cleaned_question`만 반환해도 학생 이름이 유지되는 이유는 무엇인가요?
2. builder와 compiled graph의 차이는 무엇인가요?
3. router가 반환한 `academic_route`와 `answer_academic`은 어떻게 연결되나요?
4. 현재 조건 그래프가 agent가 아니라 규칙 기반 workflow인 이유는 무엇인가요?
5. 학생지원 규칙이 단 두 단계뿐이라면 일반 파이썬 함수가 더 나을 수 있는 이유는 무엇인가요?

<details><summary>모범 답안의 핵심</summary>

1. Node 반환값은 전체 State 교체가 아니라 key별 부분 update이며 반환하지 않은 기존 key는 유지됩니다.
2. builder는 수정 가능한 설계도이고 compiled graph는 `invoke()`할 수 있는 실행 객체입니다.
3. router가 route label을 반환하고 `route_map`이 그 label을 실제 node name에 대응시킵니다.
4. 다음 행동을 모델이 아니라 사람이 작성한 `if` 규칙과 Edge가 결정하기 때문입니다.
5. 단계와 분기가 적으면 함수가 더 짧고 읽기 쉬우며 그래프 조립 비용이 불필요할 수 있기 때문입니다.
</details>

## Week 1 완료

이제 여러분은 모델의 도움 없이도 LangGraph의 기본 제어 구조를 만들고 설명할 수 있습니다.

가장 중요한 질문을 다시 기억하세요.

```text
현재 State에는 어떤 값이 있는가?
이번 Node는 무엇을 읽고 어떤 부분 update를 반환하는가?
다음 Node는 누가, 어떤 규칙으로 결정하는가?
```

이 세 질문에 답할 수 있으면 더 복잡한 workflow도 같은 방식으로 분해할 수 있습니다.